# CHB-MIT · Matched three-model training pilot
**Development-only pilot after the physics smoke test.**

Use a **GPU runtime if available** and run **Sections 1–8** in order. CPU works but full validation may take substantially longer. Your existing preprocessing run is reused; raw EDF processing is not repeated. This notebook performs actual pilot training and full inner-validation inference. It is not the final nested cross-validation experiment.

Default fit: `outer_chb01 / inner_01`. The 16 training subjects and six validation subjects remain separate; `chb01` is never loaded. All three models use the same selected training pairs, shared starting weights, batch indices and update count.

**Budget fixed before validation:** up to 1,024 unique eligible adjacent-window pairs per class, 24 examples per batch (8/class, sampled with replacement), 200 Adam updates at 0.001, gradient clipping at 5. No early stopping or checkpoint selection on validation. Three models: classifier alone; classifier + 0.5×Koopman dynamics loss; classifier + dynamics + 0.2×physics loss. Future samples are only auxiliary training targets.

This small, class-balanced pilot establishes whether the training/evaluation workflow works and gives preliminary comparisons. Its sampled prevalence differs from deployment prevalence; scores are not calibrated risks. Every validation file is replayed at its natural prevalence. We report the entire predeclared threshold grid and **do not choose a winner or operating threshold automatically**.

**Resume:** copy the printed `PILOT_DIR` into `RESUME_PILOT` in Section 3, then rerun Sections 1–8 with the same settings/runtime. Completed training updates and validation files are reused. If only a cell was interrupted and the runtime survives, rerun that cell. Training saves every 25 steps; at most the unsaved steps need repeating. Bank construction restarts if interrupted before its final save. Implementation/settings/runtime changes require a fresh pilot for a matched comparison.

## 1 · Dependencies and Drive
The bank occupies roughly 0.45 GB at the maximum pool size, plus a local copy and checkpoints. Full validation reads the saved HDF5 files; Drive I/O can dominate runtime. Progress is printed during staging, training and validation.

In [ ]:
%pip -q install numpy pandas h5py matplotlib
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import importlib.util
if importlib.util.find_spec('torch') is None:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','install','torch'])
from google.colab import drive
drive.mount('/content/gdrive')

## 2 · Frozen implementation
The embedded source strings are executed directly and hashed into the pilot plan. This makes edits detectable on resume. The same loader eligibility and 75-preceding-window availability rule are retained. Full validation encodes each valid window once per model, then forms the Top-8 mean from the last 74 preceding plus current probability. Unavailable decision scores remain NaN, while their rows remain present.

The scorer uses the same definitions as your tested scorer; interval coverage and warning occupancy are vectorized. Equivalence was checked against the original scorer. Empty-frame aggregation avoids the pandas FutureWarning. No outcome or target eligibility is used to generate alarms.

Classifier-only retains unused, frozen operator tensors for common initialization; they receive no gradients or loss. The physics variant adds eight trainable scalars. The shared encoders/classifier start identically. Rate-model bounds, equations and loss weights match the passed physics smoke test; this prior is not biologically identified and K is not guaranteed stable.

In [ ]:
SOURCE_0 = r'''
"""Fold-scoped, causal CHB-MIT loading. No model fitting or alarm scoring."""
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import h5py

class FoldData:
    def __init__(self, run, outer='outer_chb01', scope='inner_01'):
        self.run = Path(run)
        self.cfg = json.loads((self.run/'config.json').read_text())
        self.inventory = json.loads((self.run/'locked_inventory.json').read_text())
        self.events = json.loads((self.run/'locked_events.json').read_text())
        folds = json.loads((self.run/'locked_folds.json').read_text())
        self.fold = next(f for f in folds if f['id'] == outer)
        self.outer, self.scope = outer, scope
        if scope == 'outer_refit':
            self.roles = {'train': self.fold['development_subjects'], 'test': self.fold['test_subjects']}
        else:
            inner = next(f for f in self.fold['inner_folds'] if f['id'] == scope)
            self.roles = {'train': inner['train_subjects'], 'validation': inner['validation_subjects']}
        if set(self.roles['train']) & set(self.fold['test_subjects']):
            raise ValueError('Held-out subject appears in training.')
        if len(set(sum(self.roles.values(), []))) != len(sum(self.roles.values(), [])):
            raise ValueError('Roles overlap or contain duplicate subjects.')
        status = json.loads((self.run/'status.json').read_text())
        if status.get('status') not in ['review_required','preprocessing_complete']:
            raise ValueError('Preprocessing is incomplete.')
        blocking = [x for x in status.get('cohort_issues', [])
                    if x.get('reason') != 'no_valid_preictal_input_after_QC']
        if blocking:
            raise ValueError(f'Unresolved preprocessing issues: {blocking}')
        if status.get('cohort_issues'):
            audit = json.loads((self.run/'training_label_audit_status_v2_2.json').read_text())
            if audit.get('revision') != '2.2':
                raise ValueError('Run preprocessing Section 7B first.')
        self.scaler_path = self.run/'folds'/outer/scope/'train_scaler.npz'
        with np.load(self.scaler_path, allow_pickle=False) as z:
            if set(z['training_subjects'].tolist()) != set(self.roles['train']):
                raise ValueError('Scaler training subjects do not match the fold.')
            if set(z['test_subjects'].tolist()) != set(self.fold['test_subjects']):
                raise ValueError('Scaler outer test subjects do not match.')
            self.scaler = {k:z[k].copy() for k in ['raw_mean','raw_std','feature_mean','feature_std']}
        C = len(self.cfg['channels'])
        for k,shape in [('raw_mean',(C,)),('raw_std',(C,)),('feature_mean',(C,5)),('feature_std',(C,5))]:
            v = self.scaler[k]
            if v.shape != shape or not np.isfinite(v).all() or ('std' in k and np.any(v<=0)):
                raise ValueError(f'Invalid scaler: {k}')
        ratio = self.cfg['fusion_history_s']/self.cfg['window_s']
        if ratio != int(ratio):
            raise ValueError('Fusion history must contain whole windows.')
        self.history = int(ratio)
        self.subject_index = {s:i for i,s in enumerate(sorted({r['subject'] for r in self.inventory}))}
        self._metadata = {}

    def records(self, role):
        if role not in self.roles:
            raise ValueError(f'{role} is unavailable in {self.scope}; allowed: {list(self.roles)}')
        rows = [r for r in self.inventory if r['subject'] in self.roles[role]
                and r['signal_ok'] and not r['excluded_case'] and r['split']=='pool']
        return sorted(rows, key=lambda r:(r['subject'],r['case'],r['start_s'],r['relative_path']))

    def path(self, row):
        return self.run/'processed'/row['case']/'pool'/(Path(row['relative_path']).stem+'.h5')

    def metadata(self, row):
        p = self.path(row)
        if p not in self._metadata:
            with h5py.File(p,'r') as h:
                if str(h.attrs['case']) != row['case'] or json.loads(h.attrs['channels_json']) != self.cfg['channels']:
                    raise ValueError(f'Unexpected case or channels: {p}')
                if json.loads(h.attrs['config_json']) != self.cfg:
                    raise ValueError(f'Unexpected processing configuration: {p}')
                keys = ['label','valid_signal','window_start_s','decision_s','pair_with_previous']
                m = {k:h[k][:] for k in keys}
            n = len(m['label']); w = self.cfg['window_s']
            if any(len(v)!=n for v in m.values()) or not np.isin(m['label'],[-1,0,1,2]).all():
                raise ValueError(f'Invalid metadata: {p}')
            if n and (not np.allclose(m['decision_s']-m['window_start_s'],w) or
                      np.any(np.diff(m['decision_s'])<=0)):
                raise ValueError(f'Invalid timeline: {p}')
            # History ends before current window starts. Reset on every EDF boundary,
            # missing interval, or invalid signal; never concatenate across files.
            ready = np.zeros(n,dtype=bool)
            streak = 0
            for i in range(n):
                if i and not np.isclose(m['window_start_s'][i],m['decision_s'][i-1],rtol=0,atol=1e-6):
                    streak = 0
                if m['valid_signal'][i]:
                    ready[i] = streak >= self.history
                    streak += 1
                else:
                    streak = 0
            m['prediction_ready'] = ready
            self._metadata[p] = m
        return self._metadata[p]

    def index(self, role, mode='supervised'):
        if mode not in ['supervised','pairs','replay']:
            raise ValueError(mode)
        if mode=='pairs' and role!='train':
            raise ValueError('Future-window pairs are restricted to training.')
        items=[]
        for row in self.records(role):
            m=self.metadata(row); n=len(m['label'])
            if mode=='replay': keep=np.ones(n,dtype=bool)
            else:
                keep=m['prediction_ready'] & (m['label']>=0)
                if mode=='pairs':
                    next_ok=np.zeros(n,dtype=bool)
                    if n>1:
                        next_ok[:-1]=m['valid_signal'][1:] & m['pair_with_previous'][1:]
                    keep &= next_ok
            items.extend((row,int(i)) for i in np.flatnonzero(keep))
        return items

    def read(self, row, i, mode='supervised'):
        if mode not in ['supervised','pairs','replay']: raise ValueError(mode)
        m=self.metadata(row); ready=bool(m['prediction_ready'][i])
        if mode!='replay' and (not ready or m['label'][i]<0):
            raise ValueError('Supervised sample lacks labels or causal history.')
        C=len(self.cfg['channels']); L=int(self.cfg['fs']*self.cfg['window_s']); H=self.history
        inputs={'raw_t':np.zeros((C,L),np.float32), 'spec_t':np.zeros((C,5),np.float32),
                'raw_past':np.zeros((H,C,L),np.float32), 'spec_past':np.zeros((H,C,5),np.float32)}
        def normal(raw,spec):
            s=self.scaler
            x=((raw-s['raw_mean'][:,None])/s['raw_std'][:,None]).astype(np.float32)
            f=((spec-s['feature_mean'])/s['feature_std']).astype(np.float32)
            if not np.isfinite(x).all() or not np.isfinite(f).all():
                raise ValueError('Nonfinite normalized input')
            return x,f
        auxiliary={}
        with h5py.File(self.path(row),'r') as h:
            # Unavailable replay rows retain timestamps but carry zero placeholders.
            # The model must run ONLY when prediction_ready is true.
            if ready:
                inputs['raw_t'],inputs['spec_t']=normal(h['raw_uv'][i],h['features'][i])
                inputs['raw_past'],inputs['spec_past']=normal(h['raw_uv'][i-H:i],h['features'][i-H:i])
            if mode=='pairs':
                if row['subject'] not in self.roles['train']:
                    raise ValueError('Future-window pairs are restricted to training subjects.')
                if i+1>=len(m['label']) or not m['valid_signal'][i+1] or not m['pair_with_previous'][i+1]:
                    raise ValueError('Invalid adjacent pair.')
                auxiliary['raw_tdt'],auxiliary['spec_tdt']=normal(h['raw_uv'][i+1],h['features'][i+1])
        return dict(inputs=inputs, auxiliary=auxiliary,
                    target=dict(label=int(m['label'][i])),
                    metadata=dict(subject=row['subject'],case=row['case'],patient_idx=self.subject_index[row['subject']],
                                  file=str(self.path(row)),index=i,start_s=float(m['window_start_s'][i]),
                                  decision_s=float(m['decision_s'][i]),valid_signal=bool(m['valid_signal'][i]),
                                  prediction_ready=ready, reset_at_file_start=i==0))

    def iterate(self, role, mode='replay', shuffle=False, seed=20261028):
        if shuffle and (role!='train' or mode=='replay'):
            raise ValueError('Only supervised training may be shuffled.')
        items=self.index(role,mode)
        if shuffle:
            rng=np.random.default_rng(seed); rng.shuffle(items)
        for row,i in items: yield self.read(row,i,mode)

    def audit(self, role):
        rows=[];targets=[]
        for row in self.records(role):
            m=self.metadata(row)
            rows.append(dict(subject=row['subject'],file=row['relative_path'],windows=len(m['label']),
                             valid_signal=int(m['valid_signal'].sum()),prediction_ready=int(m['prediction_ready'].sum()),
                             **{f'ready_label_{k}':int((m['prediction_ready']&(m['label']==k)).sum()) for k in [-1,0,1,2]}))
        for e in self.events:
            if not e['eligible_target'] or e['split']!='pool' or e['subject'] not in self.roles[role]:continue
            a=e['onset_s']-self.cfg['sph_s']-self.cfg['sop_s'];b=e['onset_s']-self.cfg['sph_s']
            n=0
            for row in self.records(role):
                if row['case']!=e['case']:continue
                m=self.metadata(row)
                # Decision times, independent of supervised labels.
                n+=int((m['prediction_ready']&(m['decision_s']>=a)&(m['decision_s']<=b)).sum())
            targets.append(dict(event_id=e['event_id'],subject=e['subject'],retain_in_evaluation=True,
                                available_forecast_decisions=n,has_forecast_opportunity=n>0))
        return pd.DataFrame(rows),pd.DataFrame(targets)

'''
exec(compile(SOURCE_0, 'data_loading.py', 'exec'))

In [ ]:
SOURCE_1 = r'''
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

class CoreMVModel(nn.Module):
    """Explicit smoke-test foundation, not the complete physics-informed model."""
    def __init__(self, channels=18, latent=32, rank=8):
        super().__init__()
        self.raw_conv=nn.Sequential(nn.Conv1d(channels,channels,7,padding=3,groups=channels),nn.GELU(),
                                    nn.Conv1d(channels,64,1),nn.GELU())
        self.raw_project=nn.Linear(128,latent)
        self.spectral=nn.Sequential(nn.Flatten(1),nn.Linear(channels*5,64),nn.LayerNorm(64),nn.GELU(),nn.Linear(64,latent))
        self.fusion=nn.Sequential(nn.Linear(latent*2,latent),nn.LayerNorm(latent),nn.GELU())
        self.classifier=nn.Linear(latent,3)
        self.U=nn.Parameter(torch.randn(latent,rank)*0.01)
        self.V=nn.Parameter(torch.randn(latent,rank)*0.01)

    def encode(self,raw,spec):
        h=self.raw_conv(raw)
        zr=self.raw_project(torch.cat([h.mean(-1),h.amax(-1)],dim=-1))
        zs=self.spectral(spec)
        return self.fusion(torch.cat([zr,zs],dim=-1))

    def propagate(self,z):
        # Row-vector convention for K=I+UV^T: z_next = z K^T.
        return z+(z@self.V)@self.U.T

    def forward(self,raw,spec):
        z=self.encode(raw,spec)
        return self.classifier(z),z

    def operator_matrix(self):
        return torch.eye(self.U.shape[0],device=self.U.device)+self.U@self.V.T


def tensor_batch(samples,device,paired=False):
    def stack(group,key):
        return torch.as_tensor(np.stack([s[group][key] for s in samples]),dtype=torch.float32,device=device)
    out={'raw':stack('inputs','raw_t'),'spec':stack('inputs','spec_t'),
         'labels':torch.tensor([s['target']['label'] for s in samples],dtype=torch.long,device=device)}
    if paired:
        out.update(next_raw=stack('auxiliary','raw_tdt'),next_spec=stack('auxiliary','spec_tdt'))
    return out


def core_loss(model,batch,dynamics_weight=0.5):
    logits,z=model(batch['raw'],batch['spec'])
    ce=F.cross_entropy(logits,batch['labels'])
    with torch.no_grad():
        target=model.encode(batch['next_raw'],batch['next_spec'])
    dynamics=F.mse_loss(model.propagate(z),target)
    return ce+dynamics_weight*dynamics,{'classification':ce,'dynamics':dynamics},z


def check_gradients(model):
    report={}
    for name,p in model.named_parameters():
        if p.grad is None or not torch.isfinite(p.grad).all():
            raise RuntimeError(f'Missing/nonfinite gradient: {name}')
        report[name]=float(p.grad.norm().detach().cpu())
    if not any(x>0 for x in report.values()): raise RuntimeError('All gradients are zero')
    return report


@torch.no_grad()
def causal_probabilities(model, sample, device, top_k=8, encode_chunk=16):
    if not sample['metadata']['prediction_ready']:
        return None
    inputs=sample['inputs']
    # Manuscript convention: last 74 preceding windows + current = 75.
    # Loader requires 75 preceding valid windows, so its availability rule is
    # conservatively one window stricter. Preserve it; never silently relax it.
    H=len(inputs['raw_past'])
    if H<top_k or H<1: raise ValueError('Insufficient history for Top-K score')
    raw=np.concatenate([inputs['raw_past'][1:],inputs['raw_t'][None]],axis=0)
    spec=np.concatenate([inputs['spec_past'][1:],inputs['spec_t'][None]],axis=0)
    probabilities=[]
    was_training=model.training;model.eval()
    try:
        for start in range(0,len(raw),encode_chunk):
            logits,_=model(torch.as_tensor(raw[start:start+encode_chunk],device=device),
                           torch.as_tensor(spec[start:start+encode_chunk],device=device))
            probabilities.append(logits.softmax(-1).cpu())
    finally:
        model.train(was_training)
    p=torch.cat(probabilities).numpy()
    return dict(p_interictal=float(p[-1,0]),p_preictal=float(p[-1,1]),p_ictal=float(p[-1,2]),
                top8_preictal=float(np.sort(p[:,1])[-top_k:].mean()))


def smoke_sample_indices(data,per_class=2,seed=20261028):
    # Deliberately tiny class-stratified engineering test, not a training sampler.
    candidates=data.index('train','pairs');rng=np.random.default_rng(seed)
    picked=[]
    for label in [0,1,2]:
        eligible=[x for x in candidates if data.metadata(x[0])['label'][x[1]]==label]
        if not eligible: raise RuntimeError(f'No class {label} dynamics pairs in this training split')
        chosen=rng.choice(len(eligible),size=min(per_class,len(eligible)),replace=False)
        picked.extend(eligible[int(i)] for i in chosen)
    return picked

'''
exec(compile(SOURCE_1, 'model_smoke.py', 'exec'))

In [ ]:
SOURCE_2 = r'''
import math
import torch
from torch import nn
from torch.nn import functional as F

class WCInspired(nn.Module):
    """16 uncoupled formal E/I pairs sharing 8 parameters; effective seconds.
    Reduced sigmoid rate model, not the full original Wilson-Cowan equations.
    """
    def __init__(self):
        super().__init__()
        self.weight_logits=nn.Parameter(torch.logit(torch.tensor([1.5,1.,1.,0.5])/4))
        self.tau_logits=nn.Parameter(torch.logit((torch.tensor([10.,16.])-4)/36))
        self.drive_logits=nn.Parameter(torch.atanh(torch.tensor([-.5,-.5])/2))

    def physical_parameters(self):
        return 4*torch.sigmoid(self.weight_logits),4+36*torch.sigmoid(self.tau_logits),2*torch.tanh(self.drive_logits)

    def activation(self,state):
        E,I=state.chunk(2,dim=-1)
        w,tau,b=self.physical_parameters()
        return torch.cat([torch.sigmoid(w[0]*E-w[1]*I+b[0]),
                          torch.sigmoid(w[2]*E-w[3]*I+b[1])],dim=-1)

    def rhs(self,state):
        _,tau,_=self.physical_parameters()
        rate=torch.cat([tau[0].expand(state.shape[-1]//2),tau[1].expand(state.shape[-1]//2)])
        return (self.activation(state)-state)/rate

    def forward(self,state,dt_s=4.,substeps=16):
        if state.shape[-1]!=32:raise ValueError('Expected 16 E + 16 I coordinates')
        if substeps<1 or int(substeps)!=substeps or not math.isfinite(dt_s) or dt_s<=0:
            raise ValueError('Invalid integration settings')
        if not torch.isfinite(state).all() or torch.any(state<0) or torch.any(state>1):
            raise ValueError('Initial state must be finite and in [0,1]')
        _,tau,_=self.physical_parameters()
        rate=torch.cat([tau[0].expand(16),tau[1].expand(16)])
        # Exponential Euler: exact leak, activation frozen per substep.
        # Convex combination preserves [0,1] without clipping. This is NOT
        # an assertion of global dynamical stability or numerical convergence.
        decay=torch.exp(-(dt_s/int(substeps))/rate)
        y=state
        for _ in range(int(substeps)):
            y=decay*y+(1-decay)*self.activation(y)
        return y

class PhysicsMVModel(CoreMVModel):
    def __init__(self,channels=18,latent=32,rank=8):
        if latent!=32:raise ValueError('Physics v1 requires latent=32')
        super().__init__(channels,latent,rank)
        self.wc=WCInspired()


def physics_loss(model,batch,dt_s=4.,substeps=16,dynamics_weight=.5,physics_weight=.2):
    logits,z=model(batch['raw'],batch['spec'])
    with torch.no_grad():
        next_z=model.encode(batch['next_raw'],batch['next_spec'])
        next_state=torch.sigmoid(next_z)
    kz=model.propagate(z)
    wc_state=model.wc(torch.sigmoid(z),dt_s=dt_s,substeps=substeps)
    classification=F.cross_entropy(logits,batch['labels'])
    dynamics=F.mse_loss(kz,next_z)
    coupling=F.mse_loss(torch.sigmoid(kz),wc_state)
    empirical=F.mse_loss(wc_state,next_state)
    physics=(coupling+empirical)/2
    total=classification+dynamics_weight*dynamics+physics_weight*physics
    return total,dict(classification=classification,dynamics=dynamics,
                      physics_coupling=coupling,physics_empirical=empirical,physics=physics),z

@torch.no_grad()
def physics_diagnostics(model,batch,dt_s=4.):
    z=model.encode(batch['raw'],batch['spec'])
    state=torch.sigmoid(z)
    y16=model.wc(state,dt_s,16);y32=model.wc(state,dt_s,32);y256=model.wc(state,dt_s,256)
    centered=z.double()-z.double().mean(0)
    eigen=torch.linalg.svdvals(centered).square()
    if eigen.sum()>1e-20:
        p=eigen/eigen.sum();p=p[p>0];rank=float(torch.exp(-(p*p.log()).sum()).cpu())
    else:rank=0.
    K=model.operator_matrix().detach().double().cpu()
    w,tau,b=model.wc.physical_parameters()
    return dict(state_min=float(y16.min().cpu()),state_max=float(y16.max().cpu()),
                max_difference_16_vs_32=float((y16-y32).abs().max().cpu()),
                max_difference_16_vs_256=float((y16-y256).abs().max().cpu()),
                max_difference_32_vs_256=float((y32-y256).abs().max().cpu()),
                latent_mean_std=float(z.std(0,unbiased=False).mean().cpu()),
                latent_effective_rank=rank,effective_rank_upper_bound=min(len(z)-1,z.shape[-1]),
                state_saturated_fraction=float(((state<.01)|(state>.99)).float().mean().cpu()),
                koopman_spectral_radius=float(torch.linalg.eigvals(K).abs().max()),
                koopman_largest_singular_value=float(torch.linalg.svdvals(K).max()),
                wc_weights=w.cpu().tolist(),wc_tau_effective_seconds=tau.cpu().tolist(),wc_drives=b.cpu().tolist())

'''
exec(compile(SOURCE_2, 'physics_regularizer.py', 'exec'))

In [ ]:
SOURCE_3 = r'''
"""Fixed research scoring protocol v1. No training or threshold optimization."""
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd

@dataclass(frozen=True)
class ScoringPolicy:
    sph_s: float = 300.0
    sop_s: float = 1800.0
    decision_bin_s: float = 4.0
    threshold: float = 0.5  # Synthetic demonstration only; not a selected operating point.

    def __post_init__(self):
        if self.sph_s<0 or self.sop_s<=0 or self.decision_bin_s<=0 or not 0<=self.threshold<=1:
            raise ValueError('Invalid scoring policy')


def merge_intervals(intervals):
    out=[]
    for a,b in sorted((float(a),float(b)) for a,b in intervals):
        if not np.isfinite([a,b]).all() or b<=a: raise ValueError('Invalid coverage interval')
        if out and a<=out[-1][1]:out[-1]=(out[-1][0],max(out[-1][1],b))
        else:out.append((a,b))
    return out


def fully_covered(a,b,intervals):
    return any(x<=a and y>=b for x,y in intervals)


def covered_many(starts, ends, intervals):
    starts=np.asarray(starts,dtype=float);ends=np.asarray(ends,dtype=float)
    if not intervals:return np.zeros(starts.shape,dtype=bool)
    a=np.array([x[0] for x in intervals]);b=np.array([x[1] for x in intervals])
    idx=np.searchsorted(a,starts,side='right')-1
    safe=np.maximum(idx,0)
    return (idx>=0)&(b[safe]>=ends)


def score_case(predictions,events,coverage,policy=ScoringPolicy()):
    """One case, one relative-time origin. Future annotations label outcomes only.

    predictions: ALL stored decision rows, including unavailable rows. Required:
    decision_s, prediction_ready(bool), score(float, NaN when unavailable).
    events: event_id, onset_s, offset_s, eligible_target(bool).
    coverage: annotation-known intervals, including montage-unusable recordings.
    Exposure uses discrete decision bins, not interpolation over signal gaps.
    """
    p=predictions.copy();e=events.copy();obs=merge_intervals(coverage)
    required={'decision_s','prediction_ready','score'}
    if not required.issubset(p):raise ValueError(f'Missing prediction fields: {required-set(p.columns)}')
    if not {'event_id','onset_s','offset_s','eligible_target'}.issubset(e):raise ValueError('Missing event fields')
    if not p.prediction_ready.isin([True,False]).all() or not e.eligible_target.isin([True,False]).all():
        raise ValueError('Availability and eligibility must be boolean')
    p=p.sort_values('decision_s').reset_index(drop=True)
    t=p.decision_s.to_numpy(dtype=float);ready=p.prediction_ready.to_numpy(dtype=bool)
    scores=p.score.to_numpy(dtype=float)
    if not np.isfinite(t).all() or (len(t)>1 and np.any(np.diff(t)<policy.decision_bin_s-1e-6)):
        raise ValueError('Duplicate, overlapping, or invalid decision times')
    if not np.isfinite(scores[ready]).all() or np.any((scores[ready]<0)|(scores[ready]>1)):
        raise ValueError('Ready scores must be finite probabilities in [0,1]')
    if np.any(np.isfinite(scores[~ready])):raise ValueError('Unavailable scores must be NaN')
    if e.event_id.duplicated().any() or not np.isfinite(e[['onset_s','offset_s']].to_numpy(dtype=float)).all() or np.any(e.offset_s<=e.onset_s):
        raise ValueError('Invalid event table')
    if not covered_many(t-policy.decision_bin_s,t,obs).all():
        raise ValueError('Decision bin is outside annotated recording coverage')
    for row in e.itertuples():
        if not fully_covered(row.onset_s,row.offset_s,obs):
            raise ValueError('Event lies outside known annotations')
    H=policy.sph_s+policy.sop_s
    evaluable=covered_many(t,t+H,obs)&ready
    alarms=[];next_allowed=-np.inf
    # Alarm generation uses scores, readiness, time and fixed cooldown only.
    # It never reads eligibility, seizure labels or future coverage.
    for i,x in enumerate(t):
        if not ready[i] or scores[i]<policy.threshold or x<next_allowed:continue
        alarm_id=len(alarms);a=x+policy.sph_s;b=x+H
        within=e[(e.onset_s>a)&(e.onset_s<=b)]
        target_ids=within.loc[within.eligible_target.astype(bool),'event_id'].tolist()
        non_target_ids=within.loc[~within.eligible_target.astype(bool),'event_id'].tolist()
        known=fully_covered(x,b,obs)
        outcome=('target_hit' if target_ids else 'non_target_seizure' if non_target_ids else
                 'false_alarm' if known else 'unscorable')
        alarms.append(dict(alarm_id=alarm_id,alarm_s=x,sop_start_exclusive_s=a,sop_end_inclusive_s=b,
                           threshold=policy.threshold,score=float(scores[i]),outcome=outcome,
                           future_annotation_complete=known,origin_evaluable=bool(evaluable[i]),
                           early_seizure=bool(((e.onset_s>x)&(e.onset_s<=a)).any()),
                           target_ids=target_ids,non_target_ids=non_target_ids))
        next_allowed=b  # 35 minutes, including SPH, independent of observed outcomes.
    alarm_columns=['alarm_id','alarm_s','sop_start_exclusive_s','sop_end_inclusive_s','threshold','score','outcome',
                   'future_annotation_complete','origin_evaluable','early_seizure','target_ids','non_target_ids']
    alarm_table=pd.DataFrame(alarms,columns=alarm_columns)
    targets=[]
    for row in e[e.eligible_target.astype(bool)].itertuples():
        opportunities=ready&(t>=row.onset_s-H)&(t<row.onset_s-policy.sph_s)
        matches=[a for a in alarms if row.event_id in a['target_ids']]
        targets.append(dict(event_id=row.event_id,onset_s=float(row.onset_s),retained=True,
                            forecast_opportunities=int(opportunities.sum()),has_opportunity=bool(opportunities.any()),
                            predicted=bool(matches),matched_alarm_ids=[a['alarm_id'] for a in matches],
                            lead_time_s=float(row.onset_s-matches[0]['alarm_s']) if matches else np.nan))
    target_table=pd.DataFrame(targets,columns=['event_id','onset_s','retained','forecast_opportunities','has_opportunity','predicted','matched_alarm_ids','lead_time_s'])
    # Decision-bin burden: warning is active at decision t when alarm <= t < alarm+H.
    changes=np.zeros(len(t)+1,dtype=np.int64)
    for a in alarms:
        changes[np.searchsorted(t,a['alarm_s'],side='left')]+=1
        changes[np.searchsorted(t,a['sop_end_inclusive_s'],side='left')]-=1
    active=np.cumsum(changes[:-1])>0
    hits=sum(x['predicted'] for x in targets);opp=sum(x['has_opportunity'] for x in targets)
    opp_hits=sum(x['predicted'] and x['has_opportunity'] for x in targets)
    false=sum(a['outcome']=='false_alarm' for a in alarms)
    nt=sum(a['outcome']=='non_target_seizure' for a in alarms)
    eval_nt=sum(a['outcome']=='non_target_seizure' and a['origin_evaluable'] for a in alarms)
    ready_h=float(ready.sum()*policy.decision_bin_s/3600)
    eval_h=float(evaluable.sum()*policy.decision_bin_s/3600)
    summary=dict(targets=len(targets),predicted_targets=hits,targets_with_opportunity=opp,
                 sensitivity=hits/len(targets) if targets else None,
                 opportunity_conditioned_sensitivity=opp_hits/opp if opp else None,
                 recorded_decision_hours=len(t)*policy.decision_bin_s/3600,
                 prediction_ready_hours=ready_h,evaluable_decision_hours=eval_h,
                 prediction_availability=float(ready.mean()) if len(t) else None,
                 alarm_count=len(alarms),false_alarms=false,non_target_seizure_alarms=nt,
                 unscorable_alarms=sum(a['outcome']=='unscorable' for a in alarms),
                 incomplete_future_alarms=sum(not a['future_annotation_complete'] for a in alarms),
                 false_alarms_per_evaluable_hour=false/eval_h if eval_h else None,
                 non_target_alarms_at_evaluable_origins=eval_nt,
                 alarms_without_target_hit_per_evaluable_hour=(false+eval_nt)/eval_h if eval_h else None,
                 alarms_per_prediction_ready_hour=len(alarms)/ready_h if ready_h else None,
                 warning_evaluable_bins=int((active&evaluable).sum()),evaluable_bins=int(evaluable.sum()),
                 warning_ready_bins=int((active&ready).sum()),ready_bins=int(ready.sum()),
                 time_in_warning_evaluable=float((active&evaluable).sum()/evaluable.sum()) if evaluable.any() else None,
                 time_in_warning_ready=float((active&ready).sum()/ready.sum()) if ready.any() else None)
    return summary,alarm_table,target_table


def score_cohort(predictions,events,coverage_by_case,policy=ScoringPolicy()):
    if 'case' not in predictions or 'case' not in events:raise ValueError('Case IDs required')
    unknown=(set(predictions['case'])|set(events['case']))-set(coverage_by_case)
    if unknown:raise ValueError(f'Missing coverage: {unknown}')
    summaries=[];alarms=[];targets=[]
    for case,coverage in sorted(coverage_by_case.items()):
        s,a,e=score_case(predictions[predictions['case']==case],events[events['case']==case],coverage,policy)
        summaries.append(dict(case=case,**s));a.insert(0,'case',case);e.insert(0,'case',case)
        alarms.append(a);targets.append(e)
    table=pd.DataFrame(summaries)
    def total(name):return float(table[name].sum())
    def ratio(a,b):return total(a)/total(b) if total(b) else None
    eligible=table[table.targets>0]
    pooled=dict(targets=int(total('targets')),predicted_targets=int(total('predicted_targets')),
                sensitivity=ratio('predicted_targets','targets'),
                macro_case_sensitivity=float(eligible.sensitivity.mean()) if len(eligible) else None,
                false_alarms_per_evaluable_hour=ratio('false_alarms','evaluable_decision_hours'),
                time_in_warning_evaluable=ratio('warning_evaluable_bins','evaluable_bins'),
                prediction_availability=ratio('prediction_ready_hours','recorded_decision_hours'),
                unscorable_alarms=int(total('unscorable_alarms')),non_target_seizure_alarms=int(total('non_target_seizure_alarms')))
    return pooled,table,pd.DataFrame([r for a in alarms for r in a.to_dict('records')],columns=alarms[0].columns),pd.DataFrame([r for a in targets for r in a.to_dict('records')],columns=targets[0].columns)


def validate_full_prediction_export(data, role, predictions, provenance):
    """Use with FoldData from the loader notebook, before score_cohort.
    Reject incomplete previews and mismatched roles/scalers. No signal reads.
    """
    import hashlib
    if provenance.get('purpose')!='full_chronological_predictions' or provenance.get('smoke_only',True):
        raise ValueError('Smoke-test previews cannot be scored as full predictions')
    expected={'outer':data.outer,'scope':data.scope,'role':role,
              'scaler_sha256':hashlib.sha256(data.scaler_path.read_bytes()).hexdigest(),
              'folds_sha256':hashlib.sha256((data.run/'locked_folds.json').read_bytes()).hexdigest()}
    for key,value in expected.items():
        if provenance.get(key)!=value:raise ValueError('Prediction provenance mismatch: '+key)
    if not isinstance(provenance.get('model_checkpoint_sha256'),str) or len(provenance['model_checkpoint_sha256'])!=64:
        raise ValueError('Model checkpoint hash required')
    required={'relative_path','index','case','decision_s','prediction_ready','score'}
    if not required.issubset(predictions):raise ValueError('Incomplete prediction schema')
    p=predictions.copy()
    if p[['relative_path','index']].duplicated().any():raise ValueError('Duplicate prediction keys')
    rows=data.records(role);expected_paths={r['relative_path'] for r in rows}
    if set(p.relative_path)!=expected_paths:raise ValueError('Missing or unexpected recordings')
    for row in rows:
        m=data.metadata(row);part=p[p.relative_path==row['relative_path']].sort_values('index')
        n=len(m['decision_s'])
        if len(part)!=n or not np.array_equal(part['index'].to_numpy(),np.arange(n)):
            raise ValueError('Missing windows: '+row['relative_path'])
        if not (part['case']==row['case']).all() or not np.allclose(part.decision_s,m['decision_s'],rtol=0,atol=1e-6):
            raise ValueError('Case/timestamps mismatch')
        if not part.prediction_ready.isin([True,False]).all() or not np.array_equal(part.prediction_ready.to_numpy(dtype=bool),m['prediction_ready']):
            raise ValueError('Prediction availability differs from locked loader policy')
    # Include annotated recordings with unusable montage in outcome coverage.
    # Prediction availability and annotation availability are distinct concepts.
    allowed=set(data.roles[role])
    allrows=[r for r in data.inventory if r['subject'] in allowed and not r['excluded_case']]
    cases={r['case'] for r in allrows}
    coverage={case:[(r['start_s'],r['end_s']) for r in allrows if r['case']==case] for case in cases}
    events=pd.DataFrame([e for e in data.events if e['case'] in cases])
    return p,events,coverage

'''
exec(compile(SOURCE_3, 'event_scoring_fast.py', 'exec'))

In [ ]:
SOURCE_4 = r'''
import os, json, hashlib, shutil, tempfile, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import h5py
import torch
from torch.nn import functional as F

VARIANTS=('classifier','koopman','physics')

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def json_atomic(path,value):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,allow_nan=False));os.replace(tmp,path)

def init_pilot(data,destination,settings,code_sha):
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    plan=dict(stage='matched_pilot_v1',outer=data.outer,scope=data.scope,roles=data.roles,
              settings=settings,code_sha256=code_sha,scaler_sha256=digest(data.scaler_path),
              folds_sha256=digest(data.run/'locked_folds.json'),inventory_sha256=digest(data.run/'locked_inventory.json'),
              variants=list(VARIANTS),threshold_selected=False)
    if data.scope=='outer_refit':raise ValueError('Pilot restricted to development inner folds')
    p=destination/'pilot_plan.json'
    if p.exists():
        if json.loads(p.read_text())!=plan:raise ValueError('Pilot plan changed. Use a fresh pilot directory.')
    else:json_atomic(p,plan)
    return hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest()

def build_bank(data,destination,settings,fingerprint,local_root):
    destination=Path(destination);local_root=Path(local_root);local_root.mkdir(parents=True,exist_ok=True)
    saved=destination/'training_bank.h5';manifest=destination/'training_samples.json'
    if not saved.exists():
        candidates=data.index('train','pairs');rng=np.random.default_rng(settings['seed'])
        chosen=[];eligible_counts={}
        for label in [0,1,2]:
            group=[(r,i) for r,i in candidates if int(data.metadata(r)['label'][i])==label]
            eligible_counts[str(label)]=len(group)
            if not group:raise ValueError(f'No eligible training pairs for class {label}')
            pick=rng.choice(len(group),min(settings['pool_per_class'],len(group)),replace=False)
            chosen.extend((group[int(j)][0],group[int(j)][1],label) for j in pick)
        sample_manifest=[dict(relative_path=r['relative_path'],subject=r['subject'],index=i,label=k) for r,i,k in chosen]
        groups=defaultdict(list)
        for j,(r,i,k) in enumerate(chosen):groups[r['relative_path']].append((j,r,i))
        tmp=local_root/'bank_build.h5';C=len(data.cfg['channels']);L=int(data.cfg['window_s']*data.cfg['fs']);N=len(chosen)
        with h5py.File(tmp,'w') as out:
            out.attrs['fingerprint']=fingerprint
            for key,shape in [('raw',(N,C,L)),('next_raw',(N,C,L)),('spec',(N,C,5)),('next_spec',(N,C,5))]:
                out.create_dataset(key,shape,dtype='f4')
            out['labels']=np.array([k for _,_,k in chosen],dtype=np.int64)
            for fi,(relative,entries) in enumerate(sorted(groups.items())):
                row=entries[0][1];needed=sorted({idx for _,_,i in entries for idx in [i,i+1]})
                lookup={idx:j for j,idx in enumerate(needed)}
                with h5py.File(data.path(row),'r') as h:
                    raw=h['raw_uv'][needed];spec=h['features'][needed]
                s=data.scaler
                raw=((raw-s['raw_mean'][:,None])/s['raw_std'][:,None]).astype('f4')
                spec=((spec-s['feature_mean'])/s['feature_std']).astype('f4')
                if not np.isfinite(raw).all() or not np.isfinite(spec).all():raise ValueError('Nonfinite bank inputs')
                for j,_,i in entries:
                    out['raw'][j]=raw[lookup[i]];out['next_raw'][j]=raw[lookup[i+1]]
                    out['spec'][j]=spec[lookup[i]];out['next_spec'][j]=spec[lookup[i+1]]
                if fi%10==0 or fi+1==len(groups):print(f'Bank files: {fi+1}/{len(groups)}',flush=True)
        json_atomic(manifest,dict(fingerprint=fingerprint,eligible_training_pairs=eligible_counts,samples=sample_manifest))
        pending=saved.with_suffix('.h5.tmp');shutil.copyfile(tmp,pending);os.replace(pending,saved)
        json_atomic(destination/'bank_integrity.json',dict(sha256=digest(saved),fingerprint=fingerprint))
        tmp.unlink()
    if not manifest.exists():raise ValueError('Training bank manifest missing')
    info=json.loads(manifest.read_text())
    if info['fingerprint']!=fingerprint:raise ValueError('Wrong training bank')
    local=local_root/'training_bank.h5'
    integrity_path=destination/'bank_integrity.json'
    # Recover an interruption after the atomic bank copy but before its hash was saved.
    if not integrity_path.exists():json_atomic(integrity_path,dict(sha256=digest(saved),fingerprint=fingerprint))
    integrity=json.loads(integrity_path.read_text())
    if integrity['fingerprint']!=fingerprint:raise ValueError('Bank integrity mismatch')
    if not local.exists() or digest(local)!=integrity['sha256']:
        shutil.copyfile(saved,local)
    if digest(local)!=integrity['sha256']:raise ValueError('Training bank checksum mismatch')
    with h5py.File(local,'r') as h:
        if h.attrs['fingerprint']!=fingerprint:raise ValueError('Wrong bank attributes')
        bank={k:h[k][:] for k in ['raw','spec','next_raw','next_spec','labels']}
    print('Loaded bank:',len(bank['labels']),'pairs;',round(sum(v.nbytes for v in bank.values())/2**20),'MiB')
    return bank,info

def batch_indices(bank,step,settings):
    rng=np.random.default_rng(settings['seed']+100000+step)
    ids=np.concatenate([rng.choice(np.flatnonzero(bank['labels']==k),settings['batch_per_class'],replace=True) for k in [0,1,2]])
    rng.shuffle(ids);return ids

def pilot_batch(bank,ids,device):
    return {k:torch.as_tensor(v[ids],device=device,dtype=torch.long if k=='labels' else torch.float32) for k,v in bank.items()}

def new_variant(variant,channels,seed,device):
    torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    model=(PhysicsMVModel(channels=channels) if variant=='physics' else CoreMVModel(channels=channels)).to(device)
    if variant=='classifier':model.U.requires_grad_(False);model.V.requires_grad_(False)
    return model

def variant_loss(model,batch,variant):
    if variant=='physics':return physics_loss(model,batch)
    if variant=='koopman':return core_loss(model,batch)
    logits,z=model(batch['raw'],batch['spec']);ce=F.cross_entropy(logits,batch['labels'])
    return ce,dict(classification=ce),z

@torch.no_grad()
def pilot_diagnostics(model,batch,variant):
    if variant=='physics':return physics_diagnostics(model,batch)
    z=model.encode(batch['raw'],batch['spec']);center=z.double()-z.double().mean(0)
    eig=torch.linalg.svdvals(center).square();p=eig/eig.sum().clamp_min(1e-30);p=p[p>0]
    d=dict(latent_mean_std=float(z.std(0,unbiased=False).mean().cpu()),
           latent_effective_rank=float(torch.exp(-(p*p.log()).sum()).cpu()) if len(p) else 0.,
           effective_rank_upper_bound=min(len(z)-1,z.shape[-1]))
    if variant!='classifier':
        K=model.operator_matrix().double().cpu()
        d.update(koopman_spectral_radius=float(torch.linalg.eigvals(K).abs().max()),
                 koopman_largest_singular_value=float(torch.linalg.svdvals(K).max()))
    return d

def train_variant(variant,bank,settings,destination,fingerprint,device,stop_after=None):
    folder=Path(destination)/variant;folder.mkdir(exist_ok=True)
    checkpoint=folder/'last.pt'
    model=new_variant(variant,bank['raw'].shape[1],settings['seed'],device)
    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=settings['learning_rate'])
    start=0;logs=[];diagnostics=[]
    if checkpoint.exists():
        state=torch.load(checkpoint,map_location=device,weights_only=False)  # Only this pilot's own checkpoint.
        if state['fingerprint']!=fingerprint or state['variant']!=variant:raise ValueError('Checkpoint provenance mismatch')
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        start=state['step'];logs=state['logs'];diagnostics=state['diagnostics']
        print(variant,'resuming at step',start,flush=True)
    limit=min(settings['steps'],stop_after) if stop_after is not None else settings['steps']
    model.train()
    for step in range(start,limit):
        batch=pilot_batch(bank,batch_indices(bank,step,settings),device)
        optimizer.zero_grad(set_to_none=True);loss,parts,z=variant_loss(model,batch,variant)
        if not torch.isfinite(loss):raise RuntimeError(f'{variant}: nonfinite loss at step {step+1}')
        record=dict(step=step+1,total=float(loss.detach().cpu()),**{k:float(v.detach().cpu()) for k,v in parts.items()})
        checkpoint_due=(step+1)%settings['checkpoint_every']==0 or step+1==limit
        if variant=='physics' and checkpoint_due:
            w=model.raw_project.weight
            gc=torch.autograd.grad(parts['classification'],w,retain_graph=True)[0]
            gp=torch.autograd.grad(.2*parts['physics'],w,retain_graph=True)[0]
            record['weighted_physics_to_CE_gradient_ratio']=float((gp.norm()/gc.norm().clamp_min(1e-12)).detach().cpu())
        loss.backward()
        for name,p in model.named_parameters():
            if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all()):
                raise RuntimeError(f'{variant}: missing/nonfinite gradient at {name}')
        grad=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],5.)
        optimizer.step()
        if not all(torch.isfinite(p).all() for p in model.parameters()):raise RuntimeError('Nonfinite model parameter')
        record['preclip_gradient_norm']=float(grad.detach().cpu());logs.append(record)
        if checkpoint_due:
            diag=pilot_diagnostics(model,batch,variant)
            if diag['latent_mean_std']<=1e-8:raise RuntimeError('Latent variation collapsed on diagnostic batch')
            if variant=='physics' and diag['max_difference_16_vs_256']>=.005:
                raise RuntimeError('Physics integration error exceeds 0.005; review before resuming')
            diagnostics.append(dict(step=step+1,**diag))
            state=dict(fingerprint=fingerprint,variant=variant,step=step+1,model=model.state_dict(),
                       optimizer=optimizer.state_dict(),logs=logs,diagnostics=diagnostics)
            tmp=checkpoint.with_suffix('.pt.tmp');torch.save(state,tmp);os.replace(tmp,checkpoint)
            pd.DataFrame(logs).to_csv(folder/'training_log.csv',index=False)
            json_atomic(folder/'training_diagnostics.json',diagnostics)
            print(variant,'step',step+1,'/',settings['steps'],'loss',round(record['total'],5),flush=True)
    return model,logs,diagnostics

@torch.no_grad()
def infer_file(models,data,row,device,chunk=128):
    m=data.metadata(row);N=len(m['label']);prob=np.full((len(models),N,3),np.nan,dtype='f4')
    for model in models:model.eval()
    with h5py.File(data.path(row),'r') as h:
        for first in range(0,N,chunk):
            end=min(first+chunk,N);valid=m['valid_signal'][first:end]
            if not valid.any():continue
            # Read sequentially once, then use the same input for every variant.
            raw=h['raw_uv'][first:end][valid];spec=h['features'][first:end][valid];s=data.scaler
            raw=((raw-s['raw_mean'][:,None])/s['raw_std'][:,None]).astype('f4')
            spec=((spec-s['feature_mean'])/s['feature_std']).astype('f4')
            if not np.isfinite(raw).all() or not np.isfinite(spec).all():raise ValueError('Nonfinite inference inputs')
            x=torch.as_tensor(raw,device=device);f=torch.as_tensor(spec,device=device)
            ids=np.flatnonzero(valid)+first
            for j,model in enumerate(models):
                logits,_=model(x,f);p=logits.softmax(-1).cpu().numpy()
                if not np.isfinite(p).all():raise ValueError('Nonfinite predictions')
                prob[j,ids]=p
    H=data.history;scores=np.full((len(models),N),np.nan,dtype='f4')
    if H<8:raise ValueError('History shorter than Top-8')
    if N>=H:
        for j in range(len(models)):
            windows=np.lib.stride_tricks.sliding_window_view(prob[j,:,1],H)
            fused=np.partition(windows,H-8,axis=1)[:,-8:].mean(axis=1)
            scores[j,H-1:]=fused
            scores[j,~m['prediction_ready']]=np.nan
            if not np.isfinite(scores[j,m['prediction_ready']]).all():raise ValueError('Missing ready scores')
    return prob,scores

def validation_export(models,data,destination,fingerprint,device,settings):
    destination=Path(destination);folder=destination/'validation_cache';folder.mkdir(exist_ok=True)
    hashes={v:digest(destination/v/'last.pt') for v in VARIANTS}
    for v in VARIANTS:
        state=torch.load(destination/v/'last.pt',map_location='cpu',weights_only=False)
        if state['step']!=settings['steps'] or state['fingerprint']!=fingerprint:raise ValueError('Training is not complete')
    pred_id=hashlib.sha256(json.dumps(dict(pilot=fingerprint,checkpoints=hashes),sort_keys=True).encode()).hexdigest()
    records=data.records('validation');frames={v:[] for v in VARIANTS}
    for k,row in enumerate(records):
        path=folder/(row['case']+'__'+Path(row['relative_path']).stem+'.npz')
        m=data.metadata(row);cached=False
        if path.exists():
            with np.load(path,allow_pickle=False) as z:
                if str(z['fingerprint'])==pred_id and z['scores'].shape==(3,len(m['label'])):
                    prob=z['probabilities'];scores=z['scores'];cached=True
        if not cached:
            prob,scores=infer_file(models,data,row,device,settings['inference_chunk'])
            tmp=path.with_suffix('.npz.tmp')
            with tmp.open('wb') as f:np.savez_compressed(f,fingerprint=pred_id,probabilities=prob,scores=scores)
            os.replace(tmp,path)
        for j,v in enumerate(VARIANTS):
            f=pd.DataFrame(dict(relative_path=row['relative_path'],index=np.arange(len(m['label'])),case=row['case'],
                                decision_s=m['decision_s'],prediction_ready=m['prediction_ready'],score=scores[j],
                                p_interictal=prob[j,:,0],p_preictal=prob[j,:,1],p_ictal=prob[j,:,2]))
            frames[v].append(f)
        print(f'Validation files: {k+1}/{len(records)}'+(' (cached)' if cached else ''),flush=True)
    exports={}
    for v in VARIANTS:
        p=pd.concat(frames[v],ignore_index=True)
        provenance=dict(purpose='full_chronological_predictions',smoke_only=False,outer=data.outer,scope=data.scope,
                        role='validation',scaler_sha256=digest(data.scaler_path),folds_sha256=digest(data.run/'locked_folds.json'),
                        model_checkpoint_sha256=hashes[v],pilot_fingerprint=fingerprint,variant=v)
        validate_full_prediction_export(data,'validation',p,provenance)
        p.to_csv(destination/v/'validation_predictions.csv',index=False)
        json_atomic(destination/v/'prediction_provenance.json',provenance)
        exports[v]=(p,provenance)
    return exports

def window_metrics(data,predictions):
    """Descriptive classification on ready labelled windows; no class balancing."""
    confusion=np.zeros((3,3),dtype=np.int64);loss_sum=0.;total=0
    for row in data.records('validation'):
        m=data.metadata(row);p=predictions[predictions.relative_path==row['relative_path']].sort_values('index')
        probs=p[['p_interictal','p_preictal','p_ictal']].to_numpy()
        keep=m['prediction_ready']&(m['label']>=0)
        y=m['label'][keep].astype(int);pr=probs[keep]
        if len(y):
            if not np.isfinite(pr).all():raise ValueError('Nonfinite scored probabilities')
            np.add.at(confusion,(y,pr.argmax(1)),1)
            loss_sum+=float(-np.log(np.maximum(pr[np.arange(len(y)),y],1e-12)).sum());total+=len(y)
    recall=np.divide(confusion.diagonal(),confusion.sum(1),out=np.full(3,np.nan),where=confusion.sum(1)>0)
    return dict(ready_labelled_windows=total,cross_entropy=loss_sum/total if total else None,
                balanced_accuracy=float(np.nanmean(recall)) if total else None,
                class_recalls=[float(x) if np.isfinite(x) else None for x in recall],confusion_matrix=confusion.tolist())

def score_pilot(data,exports,destination,settings):
    rows=[];window={}
    for v,(p,provenance) in exports.items():
        q,e,coverage=validate_full_prediction_export(data,'validation',p,provenance)
        window[v]=window_metrics(data,p)
        for threshold in settings['threshold_grid']:
            pooled,cases,alarms,targets=score_cohort(q,e,coverage,ScoringPolicy(threshold=threshold))
            rows.append(dict(variant=v,threshold=threshold,**pooled,
                             evaluable_decision_hours=float(cases.evaluable_decision_hours.sum()),
                             prediction_ready_hours=float(cases.prediction_ready_hours.sum()),
                             total_alarms=int(cases.alarm_count.sum()),false_alarms=int(cases.false_alarms.sum()),
                             targets_with_opportunity=int(cases.targets_with_opportunity.sum()),
                             incomplete_future_alarms=int(cases.incomplete_future_alarms.sum())))
            # Save case-level results and target hits at every predeclared threshold.
            dest=Path(destination)/v/'threshold_reports';dest.mkdir(exist_ok=True)
            tag=f'{threshold:.2f}'
            cases.to_csv(dest/f'cases_{tag}.csv',index=False)
            targets.to_json(dest/f'targets_{tag}.json',orient='records',indent=2)
            alarms.to_json(dest/f'alarms_{tag}.json',orient='records',indent=2)
        print('Scoring complete:',v,flush=True)
    table=pd.DataFrame(rows);table.to_csv(Path(destination)/'validation_threshold_curves.csv',index=False)
    json_atomic(Path(destination)/'window_metrics.json',window)
    return table,window

'''
exec(compile(SOURCE_4, 'matched_pilot.py', 'exec'))

In [ ]:
IMPLEMENTATION_SHA=hashlib.sha256(('\n'.join([SOURCE_0,SOURCE_1,SOURCE_2,SOURCE_3,SOURCE_4])).encode()).hexdigest()

## 3 · Set the pilot and record the fixed plan
Keep these defaults for the first run. A fresh timestamped directory prevents overwriting earlier experiments. For a restart, paste the exact existing pilot directory into `RESUME_PILOT`. The runtime fingerprint prevents silently mixing CPU/GPU or software versions within one comparison.

In [ ]:
from datetime import datetime,timezone
RUN=Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
OUTER='outer_chb01';SCOPE='inner_03'
RESUME_PILOT=None  # Paste the full printed PILOT_DIR here after a runtime restart.
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cpu':
    torch.set_num_threads(min(4,torch.get_num_threads()))
    print('CPU selected. GPU is preferable for the full validation pass.')
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True)
SETTINGS=dict(seed=20261030,pool_per_class=1024,batch_per_class=8,steps=200,
              checkpoint_every=25,learning_rate=1e-3,inference_chunk=128,
              threshold_grid=[round(float(x),2) for x in np.linspace(0,1,21)],
              runtime=dict(torch_version=str(torch.__version__),device=str(DEVICE),
                           gpu_name=torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None))
data=FoldData(RUN,OUTER,SCOPE)
if data.cfg['window_s']!=4 or data.history!=75:
    raise RuntimeError('This pilot is specified for 4-second windows and 75 preceding history windows.')
if RESUME_PILOT is not None:
    PILOT_DIR=Path(RESUME_PILOT)
else:
    PILOT_DIR=RUN/'matched_pilot_v1'/OUTER/SCOPE/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
FINGERPRINT=init_pilot(data,PILOT_DIR,SETTINGS,IMPLEMENTATION_SHA)
LOCAL_CACHE=Path('/content/mvp_pilot_cache')/FINGERPRINT[:16]
print('PILOT_DIR =',str(PILOT_DIR))
print('Device:',DEVICE)
print('Roles:',data.roles)
print('Validation EDF files:',len(data.records('validation')))
print('Excluded outer test:',data.fold['test_subjects'])
print(json.dumps(SETTINGS,indent=2))

## 4 · Build the shared training bank
Select unique training pairs uniformly within each class (up to the cap). The pair requires valid current EEG/history, a supervised current label and a valid adjacent future window. All variants use this same pair-eligible subset, including the classifier baseline, so data differences cannot explain the comparison. No validation subjects enter the bank.

The bank contains only current and next normalized windows, not 75 copies of history per example. It is saved once to Drive, checksummed, and reused from a local copy. Direct HDF5 reads during construction can take time; the full raw dataset is not copied to Colab.

In [ ]:
bank,bank_info=build_bank(data,PILOT_DIR,SETTINGS,FINGERPRINT,LOCAL_CACHE)
counts={str(k):int((bank['labels']==k).sum()) for k in [0,1,2]}
print('Available eligible training pairs:',bank_info['eligible_training_pairs'])
print('Selected bank counts:',counts)
print('Training bank subjects:',sorted({x['subject'] for x in bank_info['samples']}))
assert not ({x['subject'] for x in bank_info['samples']}&set(data.roles['validation']))
# Verify common initial tensors before training or resuming.
initial_models=[new_variant(v,len(data.cfg['channels']),SETTINGS['seed'],'cpu') for v in VARIANTS]
shared=initial_models[0].state_dict()
for m in initial_models[1:]:
    for k in shared:torch.testing.assert_close(shared[k],m.state_dict()[k],rtol=0,atol=0)
del initial_models
print('Shared initialization verified. Batch order is determined only by seed and step.')

## 5 · Train the three variants
Runs 200 updates per variant and saves every 25. Each variant's objective is different, so compare classification losses and validation results rather than comparing total loss directly. Diagnostics are computed on checkpoint batches, not the entire cohort.

The physics model reports its weighted physics/CE gradient-norm ratio at the raw projection, helping distinguish a numerically small loss from a weak gradient effect. State bounds and numerical refinement are checked. A spectral radius above one is reported, not silently clipped. If a numerical check fails, stop and review the reported failure; do not change the tolerance within this run.

In [ ]:
trained_models=[];training_overview=[]
for variant in VARIANTS:
    model,log,diagnostics=train_variant(variant,bank,SETTINGS,PILOT_DIR,FINGERPRINT,DEVICE)
    trained_models.append(model)
    training_overview.append(dict(variant=variant,completed_steps=len(log),
                                  trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                                  final_batch_classification=log[-1]['classification'],
                                  final_batch_total=log[-1]['total']))
    display(pd.DataFrame(log).tail(3))
    print('Latest diagnostic:',json.dumps(diagnostics[-1],indent=2))
display(pd.DataFrame(training_overview))
print('All models reached the same fixed training budget. No validation checkpoint selection occurred.')

## 6 · Full chronological validation and export
Processes all 184 validation recordings for the default split. Each HDF5 chunk is read once and supplied to all three trained models. Predictions are cached per file and keyed to the exact model checkpoints, enabling resume. The live alarm cooldown later spans file boundaries within a case, while model history readiness retains the original per-file reset policy.

Every stored decision row is exported, including invalid/unavailable and unlabeled windows. A completeness validator checks subjects, files, indices, time stamps, scaler hash and availability masks before scoring. There is no held-out test inference.

In [ ]:
exports=validation_export(trained_models,data,PILOT_DIR,FINGERPRINT,DEVICE,SETTINGS)
for variant,(p,provenance) in exports.items():
    print(variant,'rows:',len(p),'ready:',int(p.prediction_ready.sum()),
          'unavailable:',int((~p.prediction_ready).sum()))
print('Full validation exports verified.')

## 7 · Descriptive validation metrics and threshold curves
The 21 thresholds (0, 0.05, …, 1) were fixed in Section 3. Report every point. No budget, threshold or winning model is selected automatically. Inner-validation results guide development; they are not unbiased held-out performance estimates.

Primary event sensitivity retains all eligible validation targets. False-alarm rates use **ready decision-bin hours with complete following 35-minute annotation coverage**, including ictal/postictal and cooldown time. Non-target-associated alarms, unknown outcomes, availability and warning burden are shown separately. This denominator is not an interictal-only FPR/h. Classification metrics use ready labeled windows only and are secondary to event scoring.

Balanced sampling is used only in training. Validation is not balanced, and probability calibration is not established. A good window-level score does not by itself establish useful forecasting.

In [ ]:
curves,classification_metrics=score_pilot(data,exports,PILOT_DIR,SETTINGS)
display(pd.DataFrame(classification_metrics).T)
display(curves)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
for variant in VARIANTS:
    c=curves[curves.variant==variant]
    axes[0].plot(c.false_alarms_per_evaluable_hour,c.sensitivity,'o-',label=variant)
    axes[1].plot(c.time_in_warning_evaluable,c.sensitivity,'o-',label=variant)
axes[0].set_xlabel('False alarms / evaluable decision hour')
axes[1].set_xlabel('Fraction of evaluable bins in warning')
for ax in axes:
    ax.set_ylabel('Eligible-target sensitivity');ax.set_ylim(-.03,1.03);ax.grid(alpha=.25);ax.legend()
fig.suptitle('Development-only pilot · full threshold grid · no operating point selected')
fig.tight_layout();fig.savefig(PILOT_DIR/'validation_tradeoffs.png',dpi=160,bbox_inches='tight')
plt.show()

## 8 · Save and share the pilot summary
Send `pilot_summary.json`, `window_metrics.json`, and `validation_threshold_curves.csv`. The summary reports completion, not success of the scientific hypothesis. Full nested evaluation, multiple seeds, uncertainty estimates and a parameter-matched generic regularizer control remain future experiments.

In [ ]:
readiness={v:dict(rows=len(p),prediction_ready=int(p.prediction_ready.sum())) for v,(p,_) in exports.items()}
summary=dict(stage='matched_pilot_v1',status='three_variants_trained_full_inner_validation_scored',
             outer=OUTER,scope=SCOPE,pilot_dir=str(PILOT_DIR),pilot_fingerprint=FINGERPRINT,
             settings=SETTINGS,training_pool_counts=counts,training_overview=training_overview,
             validation_exports=readiness,window_metrics=classification_metrics,
             validation_targets=int(curves.targets.iloc[0]),
             threshold_selected=None,model_selected=None,outer_test_accessed=False,
             interpretation='small sampled-pool development pilot; not final nested evaluation or calibrated risk',
             scorer_implementation_sha256=hashlib.sha256(SOURCE_3.encode()).hexdigest(),
             scaler_sha256=digest(data.scaler_path),folds_sha256=digest(data.run/'locked_folds.json'))
json_atomic(PILOT_DIR/'pilot_summary.json',summary)
print(json.dumps(summary,indent=2))
print('Share these files:')
for filename in ['pilot_summary.json','window_metrics.json','validation_threshold_curves.csv']:
    print(PILOT_DIR/filename)

## Troubleshooting and limits
- If Colab disconnects, preserve the pilot path and set `RESUME_PILOT` to it in Section 3. Keep settings/runtime unchanged. Training may repeat up to 24 unsaved updates; completed validation files are reused. If the bank was not yet saved, it rebuilds from the same deterministic selection.
- The default maximum bank uses approximately 0.45 GB of data in RAM, plus temporary arrays. Inference holds one chunk at a time. If a memory or deterministic-kernel error occurs, save the traceback rather than altering an active pilot's settings.
- This implementation has synthetic tests for matched initial weights, bank isolation, deterministic batches, exact CPU resume, three-variant training, causal-score equivalence, complete exports and cached validation reuse. Actual Drive throughput and GPU behavior must be established by this run.
- No performance claim follows from training loss alone. Comparisons can be unstable with 15 validation targets and a single seed. The purpose is to check learning and failure modes before spending resources on all folds.
- Keep the model definitions and integration checks explicit. The prior's eight scalars do not identify physiology, and bounded WC states do not guarantee stable Koopman dynamics. Do not increase the physics weight merely because its numeric loss is smaller; inspect its gradient contribution and development ablations.